In [ ]:

import json
import os
import requests
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

print("done")

In [ ]:

print("downloading Argo synthetic profile index")
url = "https://data-argo.ifremer.fr/argo_synthetic-profile_index.txt.gz"
df = pd.read_csv(url, skiprows=8)
df.columns = df.columns.str.strip()

df['wmo'] = df['file'].str.split('/').str[1]

regions = {
    "Arabian Sea": {"lat_min": 5, "lat_max": 20, "lon_min": 50, "lon_max": 77},
    "Bay of Bengal": {"lat_min": 5, "lat_max": 20, "lon_min": 77, "lon_max": 100},
    "Equatorial Indian Ocean": {"lat_min": -5, "lat_max": 5, "lon_min": 40, "lon_max": 100}
}

required_vars = {'TEMP', 'PSAL', 'NITRATE', 'CHLA', 'DOXY'}

print("\nrunning fleet-wide 5-variable search...")

final_verified_fleet = {}

for name, bounds in regions.items():
    print(f"scanning {name}...")
    
    spatial_mask = (
        (df['latitude'] >= bounds['lat_min']) & (df['latitude'] <= bounds['lat_max']) &
        (df['longitude'] >= bounds['lon_min']) & (df['longitude'] <= bounds['lon_max'])
    )
    region_df = df[spatial_mask].copy()

    strict_adjusted_wmos = []

    for wmo, group in region_df.groupby('wmo'):
        # check if the float structurally carries all 5 required sensors simultaneously
        five_var_profiles = group[group['parameters'].apply(
            lambda x: required_vars.issubset(set(str(x).split()))
        )]

        if five_var_profiles.empty:
            continue  # skip if the float never collected all 5 variables together

        modes_combined = "".join(five_var_profiles['parameter_data_mode'].astype(str).tolist())
        if 'D' in modes_combined or 'A' in modes_combined:
            strict_adjusted_wmos.append(wmo)

    final_verified_fleet[name] = strict_adjusted_wmos
    print(f"found {len(strict_adjusted_wmos)} floats with adjusted/delayed-mode data.")
    print(f"verified WMO IDs: {strict_adjusted_wmos}\n")

print("ready for download.")

In [ ]:

verified_fleet = {
    "Arabian Sea": ['1902457', '1902458', '1902751', '2902275', '2902306', '3902752', '3902753', '5903586'],
    "Bay of Bengal": ['1902367', '1902373', '2903829', '2903831', '5907152', '7902190'],
    "Equatorial Indian Ocean": ['1902367', '1902372', '1902453', '1902454', '1902455', '2903464', '2903465', '2903466', '2903915', '2903966', '2904013', '2904014', '4903899', '5906539', '7902190', '7902312']
}

std_depths = np.arange(0, 305, 5)
target_vars = ['temp', 'psal', 'doxy', 'chla', 'nitrate']

def fetch_float_json_data(wmo):
    base_url = "https://erddap.ifremer.fr/erddap/tabledap/ArgoFloats-synthetic-BGC.json"
    query = (
        f"?longitude,latitude,cycle_number,pres,"
        f"temp_adjusted,temp_adjusted_qc,"
        f"psal_adjusted,psal_adjusted_qc,"
        f"doxy_adjusted,doxy_adjusted_qc,"
        f"chla_adjusted,chla_adjusted_qc,"
        f"nitrate_adjusted,nitrate_adjusted_qc"
        f"&platform_number=%22{wmo}%22&pres<=300"
    )
    try:
        response = requests.get(base_url + query, timeout=30)
        if response.status_code == 200:
            json_data = response.json()
            return pd.DataFrame(json_data['table']['rows'], columns=json_data['table']['columnNames'])
    except Exception:
        pass
    return None

print("starting raw ERDDAP server JSON data processing and interpolation...")
all_processed_records = []

for region, wmos in verified_fleet.items():
    for wmo in wmos:
        print(f"pulling and processing float {wmo} in {region}...")
        raw_df = fetch_float_json_data(wmo)

        if raw_df is None or raw_df.empty:
            print(f"whoops no active records found for WMO {wmo}??")
            continue

        raw_df.columns = [col.replace('_adjusted', '').lower() for col in raw_df.columns]

        cols_to_numeric = ['longitude', 'latitude', 'cycle_number', 'pres'] + target_vars + [f"{v}_qc" for v in target_vars]
        for col in cols_to_numeric:
            if col in raw_df.columns:
                raw_df[col] = pd.to_numeric(raw_df[col], errors='coerce')

        for var in target_vars:
            if f"{var}_qc" in raw_df.columns:
                raw_df.loc[~raw_df[f"{var}_qc"].isin([1, 2]), var] = np.nan

        interpolated_runs = {var: [] for var in target_vars}
        for cycle, cycle_df in raw_df.groupby('cycle_number'):
            cycle_df = cycle_df.dropna(subset=['pres']).sort_values('pres')
            depths = cycle_df['pres'].values

            if len(depths) < 3:
                continue

            for var in target_vars:
                values = cycle_df[var].values
                valid_mask = ~np.isnan(values)
                if valid_mask.sum() > 2:
                    f_interp = interp1d(depths[valid_mask], values[valid_mask], bounds_error=False, fill_value=np.nan)
                    interpolated_runs[var].append(f_interp(std_depths))

        for idx, depth in enumerate(std_depths):
            record = {
                'region': region, 'wmo': wmo,
                'latitude': round(raw_df['latitude'].mean(), 4),
                'longitude': round(raw_df['longitude'].mean(), 4),
                'total_cycles': raw_df['cycle_number'].nunique(),
                'cycle_range': f"{raw_df['cycle_number'].min()}-{raw_df['cycle_number'].max()}",
                'depth_m': depth
            }

            for var in target_vars:
                if interpolated_runs[var]:
                    depth_slice = [run[idx] for run in interpolated_runs[var]]
                    record[var] = np.nan if np.all(np.isnan(depth_slice)) else np.nanmean(depth_slice)
                else:
                    record[var] = np.nan
            all_processed_records.append(record)

if all_processed_records:
    master_df = pd.DataFrame(all_processed_records)
    master_df.to_csv("bgc_argo_processed_profiles.csv", index=False)
    print("\data compiled and saved to: bgc_argo_processed_profiles.csv")
else:
    print("\pipeline error: no records could be compiled from the raw JSON stream")

In [ ]:

csv_filename = "bgc_argo_processed_profiles.csv"
try:
    df = pd.read_csv(csv_filename)
except FileNotFoundError:
    print(f"whoops cannot find {csv_filename}?? check your file path layout.")
    raise

var_limits = {
    'chla': (0, 1.5), 'temp': (10, 32), 'psal': (32, 37), 'doxy': (0, 300), 'nitrate': (0, 35)
}
var_colors = {
    'chla': 'forestgreen', 'temp': 'crimson', 'psal': 'navy', 'doxy': 'darkcyan', 'nitrate': 'purple'
}
var_labels = {
    'chla': 'Chl-A (mg/m³)', 'temp': 'Temp (°C)', 'psal': 'Salinity', 'doxy': 'DO (µmol/kg)', 'nitrate': 'Nitrate (µmol/kg)'
}

print("generating vertical profiles...")
df = df.sort_values(['region', 'wmo', 'depth_m'])

for (region, wmo), float_df in df.groupby(['region', 'wmo']):
    print(f"generating plot configuration for float {wmo} in {region}...")

    meta = float_df.iloc[0]
    std_depths = float_df['depth_m'].values

    fig, ax1 = plt.subplots(figsize=(8, 11))
    plt.subplots_adjust(top=0.72, bottom=0.10)

    ax1.plot(float_df['chla'], std_depths, color=var_colors['chla'], linewidth=2.5)
    ax1.set_xlabel(var_labels['chla'], color=var_colors['chla'], weight='bold', labelpad=12)
    ax1.set_xlim(var_limits['chla'])
    ax1.tick_params(axis='x', labelcolor=var_colors['chla'])

    ax1.set_ylabel('Depth (m)', weight='bold', fontsize=11)
    ax1.set_ylim(0, 300)
    ax1.invert_yaxis()
    ax1.grid(True, linestyle=':', alpha=0.5, color='gray')

    vars_to_stack = ['temp', 'psal', 'doxy', 'nitrate']
    offset_increment = 45 

    for idx, var in enumerate(vars_to_stack):
        new_ax = ax1.twiny()
        new_ax.spines['top'].set_position(('outward', offset_increment * idx))
        new_ax.plot(float_df[var], std_depths, color=var_colors[var], linewidth=2)
        new_ax.set_xlabel(var_labels[var], color=var_colors[var], weight='bold', labelpad=8)
        new_ax.set_xlim(var_limits[var])
        new_ax.tick_params(axis='x', labelcolor=var_colors[var])

        new_ax.spines['left'].set_visible(False)
        new_ax.spines['right'].set_visible(False)
        new_ax.spines['bottom'].set_visible(False)

    title_text = (
        f"Region: {region}  |  Float WMO: {wmo}\n"
        f"Mean Coordinates: {meta['latitude']}°N, {meta['longitude']}°E\n"
        f"Averaged Temporal Scope: Cycles {meta['cycle_range']} ({meta['total_cycles']} Total Cycles)"
    )
    ax1.set_title(title_text, fontsize=11, pad=190, loc='center', weight='semibold', color='black')

    plt.savefig(f"profile_wmo_{wmo}.png", dpi=300, bbox_inches='tight')
    plt.close()

print("\plotting complete. all images generated to folder.")